# Day 5 - Fine-tuning Llama-3.2-3B-Instruct with LoRA + 4-bit

Trains a LoRA adapter on the 8,292 triplets from Day 4. Concepts and the reasoning behind
every hyperparameter here: `docs/guides/04-lora-and-4bit-fine-tuning.md`.

**The baseline comes first.** Cell 6 scores the *un-fine-tuned* model on the test set before
any training happens. Our prompts hand the model the context, and an instruction-tuned 3B can
already do much of this task - so a good score after training proves nothing on its own. Only
the delta does. It also tells you what to do next: a big gain means ship it, no gain means the
dataset is the bottleneck, and a *drop* means the training config is wrong and more data
would be wasted effort.

## Before running

1. Kaggle Dataset (e.g. `finance-chatbot-dataset`) containing:
   - `train.json`, `val.json`, `test.json` - from `python -m src.dataset.build`
   - `format.py`, `metrics.py` - copied from `src/training/`
2. **Add your HF token as a Kaggle Secret named `HF_TOKEN`** (Add-ons -> Secrets). Llama-3.2
   is a gated model, so the download needs it. It is never written into the notebook.
3. Accelerator **GPU T4 x2**, Internet **On**.
4. Use **Save & Run All (Commit)** - interactive sessions idle out after ~20 minutes.

Deliberately runs on **one** T4. The 4-bit model needs ~5-7 GB, so it fits comfortably, and a
single device avoids the DataParallel-vs-device_map conflicts that make multi-GPU `Trainer`
runs fail in confusing ways.

In [ ]:
# --no-deps is load-bearing: without it pip resolves peft/bitsandbytes' torch requirement
# and downgrades Kaggle's torch, which breaks numpy and everything downstream. Their real
# dependencies (torch, transformers, accelerate) are already in the image.
!pip install -q --no-deps peft bitsandbytes

import numpy, torch, transformers
print(f"torch {torch.__version__} | numpy {numpy.__version__} | transformers {transformers.__version__}")
import peft, bitsandbytes
print(f"peft {peft.__version__} | bitsandbytes {bitsandbytes.__version__}")
print(f"CUDA {torch.cuda.is_available()}, {torch.cuda.device_count()} GPU(s)")

In [ ]:
import json, sys, time
from pathlib import Path

matches = list(Path("/kaggle/input").rglob("train.json"))
assert matches, "train.json not found under /kaggle/input - is the dataset attached?"
DATASET_DIR = matches[0].parent
OUTPUT_DIR = Path("/kaggle/working/lora-adapter")
MODEL = "meta-llama/Llama-3.2-3B-Instruct"
print("dataset dir:", DATASET_DIR)

sys.path.insert(0, str(DATASET_DIR))
from format import build_example, build_prompt
from metrics import compare, evaluate

def load(name):
    return json.loads((DATASET_DIR / f"{name}.json").read_text(encoding="utf-8"))

train_data, val_data, test_data = load("train"), load("val"), load("test")
print(f"train {len(train_data)} | val {len(val_data)} | test {len(test_data)}")
print("\nexample prompt:\n" + build_prompt(train_data[0]["question"], train_data[0]["context"]))
print("target:", train_data[0]["answer"])

In [ ]:
# Token from Kaggle Secrets - never hardcoded, never committed. This is the specific thing
# the original project got wrong (a leaked token in the notebook), so it is worth the cell.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(token=UserSecretsClient().get_secret("HF_TOKEN"))
print("logged in to Hugging Face")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.padding_side = "left"        # left for generation; the collator handles training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NF4 over the original's FP4: same memory, better
                                          # precision for normally-distributed weights
    bnb_4bit_use_double_quant=True,      # quantize the quantization constants too (~150 MB)
    bnb_4bit_compute_dtype=torch.float16, # T4 is Turing - no bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=quant_config,
    device_map={"": 0},                  # pin to one GPU, see the note above
    attn_implementation="sdpa",
)
print(f"footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
EVAL_N = 300          # of 951 test examples; enough to separate real differences, ~10 min
MAX_NEW_TOKENS = 40   # answers are capped at 25 words by the Day 4 validator

eval_set = test_data[:EVAL_N]

def predict(model, examples, batch_size=8):
    """Greedy-decode an answer for each example. Greedy, not sampled: evaluation should be
    reproducible, and we want the model's best guess rather than a sample from its tail."""
    model.eval()
    predictions = []
    for start in range(0, len(examples), batch_size):
        batch = examples[start:start + batch_size]
        prompts = [build_prompt(e["question"], e["context"]) for e in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                           truncation=True, max_length=1024).to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
        predictions += [text.strip().split("\n")[0] for text in tokenizer.batch_decode(
            out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)]
    return predictions

started = time.time()
base_predictions = predict(model, eval_set)
baseline = evaluate(base_predictions, [e["answer"] for e in eval_set])
print(f"BASELINE (no fine-tuning), {time.time() - started:.0f}s")
print(baseline)
for example, prediction in list(zip(eval_set, base_predictions))[:5]:
    print(f"\n  Q: {example['question']}\n  gold: {example['answer']}\n  pred: {prediction}")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,          # scale = alpha/r = 1.0
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # All seven projections, not just q/v: QLoRA found adapting every linear layer matters
    # more for matching full fine-tuning than a larger rank does.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # expect ~48.6M trainable, ~1.5% of the model

In [ ]:
from torch.utils.data import Dataset
from format import collate

class QADataset(Dataset):
    """Tokenizes on access, masking the prompt out of the loss (see format.build_example)."""
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        return build_example(tokenizer, self.rows[index], max_length=1024)

train_dataset, val_dataset = QADataset(train_data), QADataset(val_data)

# Sanity-check the masking before spending two hours on it: the prompt span must be -100
# and only the answer tokens should carry a real label.
sample = train_dataset[0]
trained_on = [t for t, l in zip(sample["input_ids"], sample["labels"]) if l != -100]
print(f"sequence {len(sample['input_ids'])} tokens, loss on {len(trained_on)}")
print("trained on:", repr(tokenizer.decode(trained_on)))

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints",
    num_train_epochs=2,                  # small dataset: more epochs start memorizing
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,       # effective batch 16
    learning_rate=2e-4,                  # LoRA tolerates ~10x a full fine-tune's LR, since
                                          # only the small adapter is being moved
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,
    gradient_checkpointing=True,         # ~30% slower, bounds activation memory
    optim="paged_adamw_8bit",
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=lambda batch: collate(batch, tokenizer.pad_token_id),
)

result = trainer.train()
print(result.metrics)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("adapter saved to", OUTPUT_DIR)

tuned_predictions = predict(model, eval_set)
tuned = evaluate(tuned_predictions, [e["answer"] for e in eval_set])

print("\n" + compare(baseline, tuned) + "\n")
(Path("/kaggle/working/eval-report.json")).write_text(
    json.dumps({"baseline": baseline, "tuned": tuned,
                "train_loss": result.metrics.get("train_loss"),
                "n_eval": len(eval_set)}, indent=2))

for example, before, after in list(zip(eval_set, base_predictions, tuned_predictions))[:10]:
    print(f"\nQ: {example['question']}")
    print(f"  gold : {example['answer']}")
    print(f"  base : {before}")
    print(f"  tuned: {after}")